In [2]:
# ------------------------------------------------------------
# Standard library imports
# ------------------------------------------------------------
import gc
import glob
import logging
import os
import sys
from datetime import datetime

# ------------------------------------------------------------
# Third‑party scientific stack
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe
from auxiliary_functions.time_utils import datetime64_to_yyyymmdd, string_to_yyyymm, convert_time_to_ns, convert_ns_to_datetime, extract_years_months

# ------------------------------------------------------------
# External APIs / data access
# ------------------------------------------------------------
from dateutil.relativedelta import relativedelta

# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("data_download.log", mode="a"),
        logging.StreamHandler(sys.stdout),
    ]
)
logger = logging.getLogger(__name__)

# Log uncaught exceptions to file
def log_exception(exc_type, exc_value, exc_traceback):
    if issubclass(exc_type, KeyboardInterrupt):
        # Let Ctrl+C behave normally
        sys.__excepthook__(exc_type, exc_value, exc_traceback)
        return
    logger.error("Uncaught exception", exc_info=(exc_type, exc_value, exc_traceback))

sys.excepthook = log_exception

start_date = "2020-01-01"
end_date = "2022-01-01"
logger.info(f"Start date: {start_date}")
logger.info(f"End date: {end_date}")

logger.info("Starting data download script")

def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")

logger.info(f"Dates: {np.datetime64(start_date).astype('datetime64[h]')} : {np.datetime64(end_date).astype('datetime64[h]')}")

graphcast_data_directory = f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/{string_to_yyyymm(start_date)}_{string_to_yyyymm(end_date)}"
if not os.path.exists(graphcast_data_directory):
    logger.info("Creating output directory...")
    os.makedirs(graphcast_data_directory, exist_ok=True)
logger.info(f"Graphcast data directory: {graphcast_data_directory}")

years, months = extract_years_months(start_date, end_date)

# Pressure levle variables
logger.info("Pressure Level Data")
pressure_level_base = "/gdex/data/d633000/e5.oper.an.pl"

pressure_levels = xr.DataArray(
    data = [200, 850],
    dims=['level'],
    coords={'level': [200, 850]}
)

pressure_level_variables = {
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
}

pressure_level_variables_old_names = {
    "u_component_of_wind": "U",
    "v_component_of_wind": "V",
}

for variable in pressure_level_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{pressure_level_base}/{ym}/e5.oper.an.pl.*_{pressure_level_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))

    if not files_list:
        logger.info(f"No files found for variable {variable} in month {ym}")
    else:
        logger.info(f"    Loading files...")
        pressure_level_data = convert_time_to_ns(xr.open_mfdataset(sorted(files_list), preprocess=pressure_level_preprocess).load())
        pressure_level_data = pressure_level_data.rename({pressure_level_variables_old_names[variable]: variable})

    # logger.info(pressure_level_data)
    logger.info(f"    Output directory: {graphcast_data_directory}")
    logger.info(f"    Saving data...")
    datetimes = pressure_level_data.datetime
    pressure_level_data.to_netcdf(f"{graphcast_data_directory}/{variable}.nc")
    pressure_level_data.close()
    del pressure_level_data
    gc.collect()

logger.info("Finished")

2026-07-06 10:58:05,009 [INFO] Start date: 2020-01-01
2026-07-06 10:58:05,010 [INFO] End date: 2022-01-01
2026-07-06 10:58:05,011 [INFO] Starting data download script
2026-07-06 10:58:05,014 [INFO] Dates: 2020-01-01T00 : 2022-01-01T00
2026-07-06 10:58:05,015 [INFO] Creating output directory...
2026-07-06 10:58:05,017 [INFO] Graphcast data directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/202001_202201
2026-07-06 10:58:05,017 [INFO] Pressure Level Data
2026-07-06 10:58:05,019 [INFO]   u_component_of_wind
2026-07-06 10:58:05,720 [INFO]     Loading files...
2026-07-06 13:33:56,004 [INFO]     Output directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/202001_202201
2026-07-06 13:33:56,009 [INFO]     Saving data...
2026-07-06 13:33:57,452 [INFO]   v_component_of_wind
2026-07-06 13:33:57,478 [INFO]     Loading files...
2026-07-06 16:06:48,576 [INFO]     Output directory: /glade/u/home/sressel/spencer-scratch/graphcast_input_data/cl

In [4]:
import xarray as xr
xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/climatology/202001_202201/u_component_of_wind.nc").datetime.values

array([['2020-01-01T00:00:00.000000000', '2020-01-01T06:00:00.000000000',
        '2020-01-01T12:00:00.000000000', ...,
        '2022-01-31T06:00:00.000000000', '2022-01-31T12:00:00.000000000',
        '2022-01-31T18:00:00.000000000']], dtype='datetime64[ns]')